# Colab T4 GPU Bridge Server
Automated FastMCP & REST bridge with Cloudflare Tunnel.

In [ ]:
# Cell 1: Hardware Check
import subprocess, sys
print("Checking GPU availability...")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise SystemError("FATAL: No NVIDIA GPU detected! Enable T4 GPU runtime via: Runtime -> Change runtime type -> T4 GPU.")
print(f"SUCCESS: Detected GPU -> {gpu_check.stdout.strip()}")

In [ ]:
# Cell 2: Dependencies & Cloudflared Setup
!pip install -q "fastmcp>=0.4.0" fastapi uvicorn pydantic torch
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || true
print("SUCCESS: Installed FastMCP, FastAPI, Uvicorn, and Cloudflared.")

In [ ]:
# Cell 3: Remote Worker Service (FastMCP + REST)
worker_code = '''import os, sys, subprocess, torch
from fastmcp import FastMCP
from fastapi import FastAPI
from pydantic import BaseModel

mcp = FastMCP("colab-t4-gpu")

@mcp.tool()
def gpu_info() -> dict:
    """Get current GPU specifications and VRAM utilization."""
    if not torch.cuda.is_available():
        return {"status": "error", "message": "CUDA unavailable"}
    dev = 0
    return {
        "gpu": torch.cuda.get_device_name(dev),
        "allocated_mb": round(torch.cuda.memory_allocated(dev) / (1024 ** 2), 2),
        "reserved_mb": round(torch.cuda.memory_reserved(dev) / (1024 ** 2), 2),
        "total_mb": round(torch.cuda.get_device_properties(dev).total_memory / (1024 ** 2), 2)
    }

@mcp.tool()
def run_gpu_command(cmd: str) -> dict:
    """Execute a shell command with GPU acceleration on the Colab instance."""
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return {
        "returncode": res.returncode,
        "stdout": res.stdout,
        "stderr": res.stderr
    }

@mcp.tool()
def run_inference(prompt: str) -> dict:
    """Generic GPU inference endpoint template."""
    return {"prompt": prompt, "status": "success", "device": torch.cuda.get_device_name(0)}

class CmdRequest(BaseModel):
    cmd: str

class InferRequest(BaseModel):
    prompt: str

app = mcp.get_app() if hasattr(mcp, 'get_app') else FastAPI()

@app.get("/health")
def health():
    return {"status": "ok", "cuda": torch.cuda.is_available(), "info": gpu_info()}

@app.post("/api/run")
def api_run(req: CmdRequest):
    return run_gpu_command(req.cmd)

@app.post("/api/infer")
def api_infer(req: InferRequest):
    return run_inference(req.prompt)
'''

with open('remote_worker.py', 'w') as f:
    f.write(worker_code)
print("SUCCESS: Generated remote_worker.py")

In [ ]:
# Cell 4: Launch Server & Cloudflare Tunnel
import time, re, subprocess

!pkill -f 'uvicorn remote_worker' || true
!pkill -f 'cloudflared tunnel' || true

server_proc = subprocess.Popen(["uvicorn", "remote_worker:app", "--host", "0.0.0.0", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

if tunnel_url:
    print("=" * 60)
    print("COLAB T4 GPU TUNNEL READY!")
    print(f"ENDPOINT URL: {tunnel_url}")
    print(f"SSE MCP URL:  {tunnel_url}/sse")
    print("=" * 60)
    print(f"Copy this URL and run locally:\n  python connect_colab.py --url {tunnel_url}")
else:
    print("ERROR: Cloudflare tunnel generation timed out. Check output.")